# Configurable Small-Sample Corrections (SSC) — polars_reg v0.3.0

**What's new in v0.3.0:**

- **Configurable SSC** — `ssc()` function controls degrees-of-freedom adjustments for variance-covariance matrices. Match any package's conventions: pyfixest (default), Stata `reghdfe`, Stata `ivregress`, R `fixest`, or turn off all corrections.
- **Panel wrappers share VCV dispatch** — `panel_fe()` is now syntactic sugar around `ols()` with absorbed FE, inheriting the full SSC configuration.
- **`compare()` with `match_ssc=True`** — When comparing against other packages, polars_reg can automatically re-run with each backend's SSC conventions so that SE differences due to dfc choices are isolated from numerical differences.

This notebook demonstrates each SSC option and shows how to match different packages' standard errors exactly.

---

## Setup

In [ ]:
import numpy as np
import polars as pl
import polars_reg as pr
from polars_reg import ssc, SSC

---

## Simulated Data

Panel dataset with known parameters:

| Parameter | True value |
|-----------|------------|
| β₁ (x1) | 2.0 |
| β₂ (x2) | -0.5 |
| Intercept | 1.0 |
| Entity FE | N(0, 0.5²) across 50 firms |
| Error | N(0, 0.5²) |

In [ ]:
rng = np.random.default_rng(42)
n_firms, n_years = 50, 20
n = n_firms * n_years

# Panel structure
firm_id = np.repeat(np.arange(n_firms), n_years)
year_id = np.tile(np.arange(2000, 2000 + n_years), n_firms)

# Entity fixed effects
fe = rng.standard_normal(n_firms) * 0.5

# Regressors
x1 = rng.standard_normal(n)
x2 = rng.standard_normal(n)

# Error
u = rng.standard_normal(n) * 0.5

# Outcome: y = 1 + 2*x1 - 0.5*x2 + firm_fe + error
y = 1.0 + 2.0 * x1 - 0.5 * x2 + fe[firm_id] + u

# Cluster variable (10 industries, each containing 5 firms)
industry = np.repeat(np.arange(10), n_years * 5)

df = pl.DataFrame({
    "y": y, "x1": x1, "x2": x2,
    "firm_id": firm_id, "year_id": year_id,
    "industry": industry,
})
print(f"Panel: {n_firms} firms \u00d7 {n_years} years = {n} observations")
df.head(5)

---

## 1. Default SSC (pyfixest convention)

By default, polars_reg uses the same small-sample corrections as pyfixest:

- `k_adj=True` — residual df scaling: `(N-1)/(N-k)` for clustered, `N/(N-k)` for robust
- `k_fixef="none"` — absorbed FE do **not** count toward k
- `G_adj=True` — cluster scaling: `G/(G-1)`
- `G_df="conventional"` — each cluster dimension gets its own G/(G-1) adjustment

This is equivalent to `ssc()` with no arguments:

In [ ]:
# These two calls are identical:
result_default = pr.ols("y ~ x1 + x2 | firm_id", df, cluster=["firm_id"])
result_explicit = pr.ols("y ~ x1 + x2 | firm_id", df, cluster=["firm_id"], ssc=ssc())

print("Default SSC:", SSC())  # shows the default values
print()
result_default.summary()

In [ ]:
# Verify they're identical
assert np.allclose(result_default.se, result_explicit.se), "Should be identical"
print("Default and explicit ssc() produce identical SEs: confirmed")

---

## 2. Matching Stata reghdfe

Stata's `reghdfe` uses different conventions from pyfixest:

- `k_fixef="nonnested"` — FE dimensions **not nested** in any cluster count toward k in the dfc
- `G_df="min"` — for multiway clustering, use `min(G)/(min(G)-1)` across all cluster dimensions

The preset is `ssc(k_fixef="nonnested", G_df="min")`:

In [ ]:
result_pyfixest = pr.ols("y ~ x1 + x2 | firm_id", df, cluster=["firm_id"])
result_reghdfe = pr.ols(
    "y ~ x1 + x2 | firm_id", df,
    cluster=["firm_id"],
    ssc=ssc(k_fixef="nonnested", G_df="min"),
)

print("=== pyfixest convention (default) ===")
print(f"  x1 SE: {result_pyfixest.se[0]:.6f}")
print(f"  x2 SE: {result_pyfixest.se[1]:.6f}")
print()
print("=== Stata reghdfe convention ===")
print(f"  x1 SE: {result_reghdfe.se[0]:.6f}")
print(f"  x2 SE: {result_reghdfe.se[1]:.6f}")
print()
print("SE ratio (reghdfe / pyfixest):")
for i, name in enumerate(result_reghdfe.names):
    ratio = result_reghdfe.se[i] / result_pyfixest.se[i]
    print(f"  {name}: {ratio:.6f}")
print()
print("Note: When firm_id FE is nested in firm_id cluster, k_fixef='nonnested'")
print("has no effect (no non-nested FE). The difference comes from G_df in multiway cases.")

Now consider a case where the FE is **not** nested in the cluster (clustering by industry instead of firm):

In [ ]:
# firm_id FE is NOT nested in industry cluster
# (each industry contains multiple firms)
result_pf = pr.ols("y ~ x1 + x2 | firm_id", df, cluster=["industry"])
result_rh = pr.ols(
    "y ~ x1 + x2 | firm_id", df,
    cluster=["industry"],
    ssc=ssc(k_fixef="nonnested", G_df="min"),
)

print("Clustering by INDUSTRY (firm FE not nested in cluster):")
print()
print("=== pyfixest convention (k_fixef='none') ===")
print(f"  x1 SE: {result_pf.se[0]:.6f}")
print(f"  x2 SE: {result_pf.se[1]:.6f}")
print()
print("=== Stata reghdfe convention (k_fixef='nonnested') ===")
print(f"  x1 SE: {result_rh.se[0]:.6f}")
print(f"  x2 SE: {result_rh.se[1]:.6f}")
print()
print("SE ratio (reghdfe / pyfixest):")
for i, name in enumerate(result_rh.names):
    ratio = result_rh.se[i] / result_pf.se[i]
    print(f"  {name}: {ratio:.6f}")
print()
print("Here the ratios differ from 1.0 because reghdfe counts non-nested FE")
print("in the dfc denominator, making the correction factor larger.")

---

## 3. Matching Stata ivregress (asymptotic)

Stata's `ivregress` uses asymptotic variance formulas with **no** small-sample corrections:

- `k_adj=False` — no residual df scaling (use N, not N-k, in denominator)
- `G_adj=False` — no cluster scaling (omit G/(G-1))

The preset is `ssc(k_adj=False, G_adj=False)`:

In [ ]:
# Simulate IV data
z1_iv = rng.standard_normal(n)
z2_iv = rng.standard_normal(n)
u_iv = rng.standard_normal(n) * 0.5
x_endog = 0.5 * z1_iv + 0.3 * z2_iv + 0.4 * u_iv
y_iv = 1.0 + 2.0 * x1 - 0.5 * x2 + 0.8 * x_endog + u_iv

df_iv = df.with_columns(
    pl.Series("y_iv", y_iv),
    pl.Series("x_endog", x_endog),
    pl.Series("z1", z1_iv),
    pl.Series("z2", z2_iv),
)

In [ ]:
# Default SSC (pyfixest)
result_iv = pr.iv2sls("y_iv ~ x1 + x2 || x_endog ~ z1 + z2", df_iv)

# Stata ivregress convention (asymptotic, no corrections)
result_iv_stata = pr.iv2sls(
    "y_iv ~ x1 + x2 || x_endog ~ z1 + z2", df_iv,
    ssc=ssc(k_adj=False, G_adj=False),
)

print("=== Default (pyfixest) — iid SEs ===")
for name, se_val in zip(result_iv.names, result_iv.se):
    print(f"  {name:>10}: SE = {se_val:.6f}")

print()
print("=== Stata ivregress — asymptotic SEs ===")
for name, se_val in zip(result_iv_stata.names, result_iv_stata.se):
    print(f"  {name:>10}: SE = {se_val:.6f}")

print()
print("The difference comes from sigma^2 = e'e/(N-k) vs e'e/N.")
print(f"Ratio sqrt(N/(N-k)) = {np.sqrt(n / (n - len(result_iv.names))):.6f}")

---

## 4. No Corrections At All

For theoretical work or custom corrections, you can turn off all dfc adjustments:

In [ ]:
result_adj = pr.ols("y ~ x1 + x2", df, cluster=["industry"])
result_none = pr.ols(
    "y ~ x1 + x2", df,
    cluster=["industry"],
    ssc=ssc(k_adj=False, G_adj=False),
)

print("=== With corrections (default) ===")
for name, se_val in zip(result_adj.names, result_adj.se):
    print(f"  {name:>5}: SE = {se_val:.6f}")

print()
print("=== No corrections ===")
for name, se_val in zip(result_none.names, result_none.se):
    print(f"  {name:>5}: SE = {se_val:.6f}")

print()
n_obs = result_adj.n_obs
k_est = len(result_adj.names)
G = 10  # industries
expected_ratio = np.sqrt((G / (G - 1)) * (n_obs - 1) / (n_obs - k_est))
print(f"Expected SE ratio = sqrt(G/(G-1) * (N-1)/(N-k)):")
print(f"  = sqrt({G}/{G-1} * {n_obs-1}/{n_obs-k_est}) = {expected_ratio:.6f}")
print(f"Actual ratio (x1): {result_adj.se[0] / result_none.se[0]:.6f}")

---

## 5. Panel FE as ols() Wrapper

`panel_fe()` is syntactic sugar for `ols()` with absorbed entity FE. Both accept `ssc=` and produce identical results:

In [ ]:
# panel_fe() with default cluster=[entity]
r1 = pr.panel_fe("y ~ x1 + x2", df, entity="firm_id", cluster=["firm_id"])

# Equivalent ols() call
r2 = pr.ols("y ~ x1 + x2 | firm_id", df, cluster=["firm_id"])

print("panel_fe() vs ols() with absorbed FE:")
print()
for i, name in enumerate(r1.names):
    print(f"  {name}:")
    print(f"    panel_fe  coef={r1.coefficients[i]:.8f}  SE={r1.se[i]:.8f}")
    print(f"    ols       coef={r2.coefficients[i]:.8f}  SE={r2.se[i]:.8f}")

print()
assert np.allclose(r1.coefficients, r2.coefficients), "Coefficients differ!"
assert np.allclose(r1.se, r2.se), "SEs differ!"
print("Coefficients and SEs are identical: confirmed")

In [ ]:
# SSC passes through panel_fe() to ols()
r_stata = pr.panel_fe(
    "y ~ x1 + x2", df,
    entity="firm_id", cluster=["firm_id"],
    ssc=ssc(k_fixef="nonnested", G_df="min"),
)

r_ols_stata = pr.ols(
    "y ~ x1 + x2 | firm_id", df,
    cluster=["firm_id"],
    ssc=ssc(k_fixef="nonnested", G_df="min"),
)

assert np.allclose(r_stata.se, r_ols_stata.se)
print("SSC forwarding through panel_fe() confirmed:")
print(f"  panel_fe SE: {r_stata.se}")
print(f"  ols      SE: {r_ols_stata.se}")

---

## 6. SSC with compare()

The `compare()` function runs the same regression across multiple backends. With `match_ssc=True`, it also re-runs polars_reg using each backend's SSC conventions, so you can isolate SE differences caused by dfc choices from numerical algorithm differences.

In [ ]:
# Basic comparison with HC1 SEs
report = pr.compare("ols", "y ~ x1 + x2", df, vcov="HC1", backend="pyfixest")
report.summary()

In [ ]:
# With match_ssc=True: adds polars_reg columns with per-backend SSC
report_matched = pr.compare(
    "ols", "y ~ x1 + x2 | firm_id", df,
    cluster=["firm_id"],
    match_ssc=True,
    backend=["pyfixest", "statsmodels"],
    rtol=1e-4,
)
report_matched.summary()

In [ ]:
# See the equivalent code for each run
print(report_matched.code())

---

## 7. Side-by-Side regtable() with Different SSC

Use `regtable()` to compare the same regression under different SSC configurations:

In [ ]:
r_default = pr.ols("y ~ x1 + x2 | firm_id", df, cluster=["firm_id"])
r_reghdfe = pr.ols(
    "y ~ x1 + x2 | firm_id", df,
    cluster=["firm_id"],
    ssc=ssc(k_fixef="nonnested", G_df="min"),
)
r_nocorr = pr.ols(
    "y ~ x1 + x2 | firm_id", df,
    cluster=["firm_id"],
    ssc=ssc(k_adj=False, G_adj=False),
)

pr.regtable(
    r_default, r_reghdfe, r_nocorr,
    labels=["pyfixest SSC", "Stata reghdfe SSC", "No corrections"],
).tab_header(
    title="Effect of SSC on Standard Errors",
    subtitle="Same model, different small-sample corrections",
)

---

## 8. Common Presets Reference

| Package / Command | `ssc()` call | k_adj | k_fixef | G_adj | G_df |
|---|---|---|---|---|---|
| **pyfixest** (default) | `ssc()` | True | `"none"` | True | `"conventional"` |
| **Stata reghdfe** | `ssc(k_fixef="nonnested", G_df="min")` | True | `"nonnested"` | True | `"min"` |
| **R fixest** | `ssc(k_fixef="nonnested")` | True | `"nonnested"` | True | `"conventional"` |
| **Stata ivregress** | `ssc(k_adj=False, G_adj=False)` | False | `"none"` | False | `"conventional"` |
| **No corrections** | `ssc(k_adj=False, G_adj=False)` | False | `"none"` | False | `"conventional"` |
| **Full FE counting** | `ssc(k_fixef="full")` | True | `"full"` | True | `"conventional"` |

### Parameter details

- **`k_adj`** (`bool`) — Whether to apply residual degrees-of-freedom scaling. `True`: `(N-1)/(N-k)` for clustered, `N/(N-k)` for robust. `False`: no scaling (asymptotic).

- **`k_fixef`** (`str`) — How absorbed FE count in `k`:
  - `"none"`: FE excluded from k (pyfixest default)
  - `"nonnested"`: Only FE not nested in any cluster dimension count in k (Stata reghdfe, R fixest)
  - `"full"`: All absorbed FE parameters count in k

- **`G_adj`** (`bool`) — Whether to apply `G/(G-1)` cluster scaling.

- **`G_df`** (`str`) — For multiway clustering:
  - `"conventional"`: Each term in the Cameron-Gelbach-Miller sum gets its own `G_i/(G_i-1)`
  - `"min"`: All terms use `min(G)/(min(G)-1)` (Stata reghdfe convention)

In [ ]:
# Quick reference: print each preset
presets = {
    "pyfixest (default)": ssc(),
    "Stata reghdfe":      ssc(k_fixef="nonnested", G_df="min"),
    "R fixest":           ssc(k_fixef="nonnested"),
    "Stata ivregress":    ssc(k_adj=False, G_adj=False),
    "No corrections":     ssc(k_adj=False, G_adj=False),
    "Full FE counting":   ssc(k_fixef="full"),
}

for name, preset in presets.items():
    print(f"{name:25s}  {preset}")